In [ ]:
import scanpy as sc
import anndata as ad
import pandas as pd
import warnings; warnings.simplefilter('ignore')
import spatialdata as sd
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import wilcoxon, chi2_contingency, mannwhitneyu
from scipy.sparse import block_diag
from statsmodels.stats.multitest import multipletests
from itertools import combinations
import senepy as sp
import scipy.sparse as sparse
from typing import Optional, Tuple, Literal
import math
import squidpy as sq

In [ ]:
import sys
from pathlib import Path

_p = Path.cwd().resolve()
while not (_p / 'config.yaml').exists() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p / 'scripts'))
from paths import P, ensure_dirs

In [ ]:
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 12,
})
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

# Data loading

In [ ]:
ADATA_PATH  = str(P.processed.adata.all_cells / P.fn.all_cells_final)
SDATA_DIR   = str(P.interim.sdata)

all_cells_adata = sc.read_h5ad(ADATA_PATH)
epi_adata = sc.read_h5ad(str(P.processed.adata.epithelial / "epithelial_20_50_harmony_batch_05_pt_05_ssg.h5ad"))
all_cells_adata = all_cells_adata[~all_cells_adata.obs['cell_id'].duplicated(keep=False)].copy()

In [ ]:
cluster_colors = {
    "0": '#009432',
    "1": '#C4E538',
    "2": "#FF788F",
    "3": "#85D5FB",
    "4": "#B3A1F9",
    "5": '#0652DD',
    "6": '#F79F1F',
    "7": "#a53707",
    "8": '#833471',
    "9": '#EA2027',
    "10": '#1B1464'
}

colors_dict = {'B-Cells': '#009432ff',
 'Endothelial': "#521243ff",
 'Fibroblasts': "#ff9090",
 'Mast Cells': "#A9422A",
 'Myeloid Cells': '#006266ff',
 'Pericytes': '#81fcf8',
 'Plasma Cells': '#f79f1fff',
 'Schwann Cells': '#8643cc',
 'Smooth Muscle': '#1518c2',
 'T-Cells': '#c4e538ff',
 'CD8-T-Cell':"#EA2027",
 'Epithelial': '#34ace0'}

## Spatialdata prep

In [ ]:
def make_spatialdata_dict(spatialdata_dir):
    spatialdata_dict = {}
    entries = [f for f in os.listdir(spatialdata_dir) if f.startswith("fil")]
    for filename in sorted(entries):
        sdata = sd.read_zarr(f"{spatialdata_dir}/{filename}")
        key = filename.replace('filtered_normalized_xenium_', '').replace('.zarr', '')
        spatialdata_dict[key] = sdata
    return spatialdata_dict


def transfer_obs_columns(spatialdata_dict, reference_adata, columns):
    ref_obs = reference_adata.obs.set_index('cell_id')[columns]
    for section_key, sdata in spatialdata_dict.items():
        table = sdata.tables['table']
        for col in columns:
            if col in ref_obs.columns:
                table.obs[col] = table.obs['cell_id'].map(ref_obs[col])
    return spatialdata_dict


spatialdata_dict = make_spatialdata_dict(SDATA_DIR)
print(f"Loaded {len(spatialdata_dict)} SpatialData sections")

In [ ]:
MIN_EPI_CELLS = 100

epi_obs = all_cells_adata.obs[all_cells_adata.obs['annotation_final_fine_cd8'] == 'Epithelial']
epi_counts = epi_obs.groupby(['batch', 'core_id']).size()
valid_pairs = epi_counts[epi_counts >= MIN_EPI_CELLS].reset_index()
valid_section_cores = set(zip(valid_pairs['batch'], valid_pairs['core_id']))
print(f"Valid (section, core) pairs: {len(valid_section_cores)}")

valid_cell_ids = set(all_cells_adata.obs['cell_id'].unique())

filtered_spatialdata_dict = {}
for section_key, sdata in spatialdata_dict.items():
    table = sdata.tables['table']
    valid_cores_for_section = {
        core for (batch, core) in valid_section_cores if batch == section_key
    }
    mask = (
        table.obs['cell_id'].isin(valid_cell_ids)
        & table.obs['core_id'].isin(valid_cores_for_section)
    )
    if mask.sum() > 0:
        sdata.tables['table'] = table[mask].copy()
        filtered_spatialdata_dict[section_key] = sdata
        print(f"  {section_key}: {mask.sum()} cells, "
              f"{sdata.tables['table'].obs['core_id'].nunique()} cores")
    else:
        print(f"  {section_key}: dropped (no valid cells)")

core_to_sections = {}
for section_key, sdata in filtered_spatialdata_dict.items():
    for core in sdata.tables['table'].obs['core_id'].unique():
        core_to_sections.setdefault(core, []).append(section_key)
duplicated = {c: s for c, s in core_to_sections.items() if len(s) > 1}
print(f"\nCores in multiple sections: {len(duplicated)}")
for core, sections in sorted(duplicated.items()):
    print(f"  {core}: {sections}")

In [ ]:
TRANSFER_COLUMNS = [
    'annotation_final_fine', 'annotation_final_coarse',
    'tissue_type_cell_level', 'tissue_type_dysplasia_cell_level',
    'mixed_core_tissue_type', 'mixed_core_dysplasia',
    'tissue_type_cell_level_normal_split',
    'tissue_type_dysplasia_cell_level_normal_split',
    'epithelial_0.8',
    'annotation_final_fine_cd8',
    'GDF15_more_than_1_transcript', 'senepy_intestine_epi_0',
    'stemness_score', 'senepy_high', 'core_id', 'patient_id',
]

filtered_spatialdata_dict = transfer_obs_columns(
    filtered_spatialdata_dict, all_cells_adata, TRANSFER_COLUMNS
)

for section_key, sdata in filtered_spatialdata_dict.items():
    table = sdata.tables['table']
    n_epi = (table.obs['annotation_final_fine_cd8'] == 'Epithelial').sum()
    n_total = len(table)
    print(f"{section_key}: {n_total} total, {n_epi} epithelial ({100*n_epi/n_total:.1f}%)")

## All cells UMAPs and marker dotplots

In [ ]:
output_dir = str(P.results.figures / "figure_5")
os.makedirs(output_dir, exist_ok=True)


In [ ]:
all_cells_adata.obs["annotation_final_fine_cd8_combined"] = all_cells_adata.obs["annotation_final_fine_cd8"].astype(str).copy()

fib_mask = all_cells_adata.obs["annotation_final_fine_cd8_combined"].str.contains("Fibroblast|CAF", case=False, na=False)
all_cells_adata.obs.loc[fib_mask, "annotation_final_fine_cd8_combined"] = "Fibroblasts"
adata_filtered = all_cells_adata[all_cells_adata.obs["annotation_final_fine_cd8_combined"] != "Unknown"].copy()

category_order = [
    "Epithelial",
    "Fibroblasts",
    "Smooth Muscle",
    "Endothelial",
    "Pericytes",
    "Schwann Cells",
    "Plasma Cells",
    "B-Cells",
    "T-Cells",
    "CD8-T-Cell",
    "Myeloid Cells",
    "Mast Cells"
]

final_markers = ["EPCAM","LGALS3","TSPAN8","COL1A1","DCN","CXCL14","ACTA2","IGFBP5","IGFBP4","PLVAP","PECAM1","IGFBP7","COL4A1","MPZ","SCN7A","XBP1","MZB1","TENT5C","MS4A1","CD79A","PTPRC","CD3E","CD8A","CTSB","KIT","MS4A2"]
existing = set(adata_filtered.obs["annotation_final_fine_cd8_combined"].unique())
category_order = [c for c in category_order if c in existing]

In [ ]:
dp = sc.pl.dotplot(
    adata_filtered,
    var_names=final_markers,
    groupby="annotation_final_fine_cd8_combined",
    categories_order=category_order,
    standard_scale="var",
    return_fig=True,
    swap_axes=True,
)
dp.style(dot_edge_color="none")
ax_dict = dp.show(return_axes=True)

ax_dict["mainplot_ax"].tick_params(axis="both", labelsize=14)
ax_dict["mainplot_ax"].set_xticklabels(
    ax_dict["mainplot_ax"].get_xticklabels(), rotation=45, ha="right", fontsize=14
)

for ax_name, ax in ax_dict.items():
    if ax_name != "mainplot_ax":
        ax.tick_params(labelsize=12)
        if ax.get_title():
            ax.set_title(ax.get_title(), fontsize=14)
plt.savefig(os.path.join(output_dir, "dotplot.pdf"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
adata_filtered.obs["annotation_collapsed"] = adata_filtered.obs["annotation_final_fine_cd8"].copy()

sc.settings._vector_friendly = True

ax = sc.pl.umap(adata_filtered, color="annotation_collapsed", show=False, size=0.5, title=None, legend_loc="none", palette=colors_dict)
ax.set_title("")
ax.set_axis_off()
plt.savefig(os.path.join(output_dir, "all_cells_umap.pdf"), dpi=300, bbox_inches="tight")
plt.show()

sc.settings._vector_friendly = False

In [ ]:
obs = adata_filtered.obs.copy()

epi_mask = obs['annotation_final_fine'] == 'Epithelial'
obs['combined_tissue_type_colors_plotting'] = ''

obs.loc[epi_mask, 'combined_tissue_type_colors_plotting'] = (
    obs.loc[epi_mask, 'tissue_type_cell_level_normal_split'].astype(str)
)

epi_obs = obs.loc[epi_mask]
core_tissue = (
    epi_obs.loc[epi_obs['tissue_type_cell_level_normal_split'].notna()]
    .groupby('core_id')['tissue_type_cell_level_normal_split']
    .apply(lambda x: x.dropna().unique())
)

core_label = core_tissue.apply(lambda x: x[0] if len(x) == 1 else 'Mixed')

non_epi_mask = ~epi_mask
obs.loc[non_epi_mask, 'combined_tissue_type_colors_plotting'] = (
    obs.loc[non_epi_mask, 'core_id'].map(core_label)
)

adata_filtered.obs = obs

print(adata_filtered.obs['combined_tissue_type_colors_plotting'].value_counts())

In [ ]:
tissue_colors_normal_split_fig1 = {'Dist_N': "#3dcd55",'Adj_N': '#40407a', 'AD': '#ffb142', 'CA': '#fa2b2b', "Mixed":"#ddc2fc"}
sc.settings._vector_friendly = True
ax = sc.pl.umap(adata_filtered, color="combined_tissue_type_colors_plotting", show=False, size=0.5, title=None, legend_loc="none", palette=tissue_colors_normal_split_fig1)
ax.set_title("")
ax.set_axis_off()
plt.savefig(os.path.join(output_dir, "all_cells_umap_by_tissue_type.pdf"), dpi=300, bbox_inches="tight")
plt.show()
sc.settings._vector_friendly = False

## Epithelial UMAPs

In [ ]:
tissue_colors_normal_split_fig1_epi = {'Dist_N': "#3dcd55",'Adj_N': '#40407a', 'AD': '#ffb142', 'CA': "#fa2b2b"}
sc.settings._vector_friendly = True

ax = sc.pl.umap(epi_adata, color="epithelial_0.8", show=False,size=0.8,title=None, legend_loc="none", palette=cluster_colors)
ax.set_title("")
ax.set_axis_off()
plt.savefig(os.path.join(output_dir,"epi_umap.pdf"), dpi=300, bbox_inches="tight")
plt.show()

ax = sc.pl.umap(epi_adata, color="tissue_type_cell_level_normal_split", show=False,size=0.8,title=None, legend_loc="none", palette=tissue_colors_normal_split_fig1_epi)
ax.set_title("")
ax.set_axis_off()
plt.savefig(os.path.join(output_dir, "epi_umap_by_tissue_type.pdf"), dpi=300, bbox_inches="tight")
plt.show()
sc.settings._vector_friendly = False

In [ ]:
fig_leg, ax_leg = plt.subplots(1, 1, figsize=(4, 2))
ax_leg.axis("off")

handles = [mpatches.Patch(color=tissue_colors_normal_split_fig1_epi[k], label=k) for k in tissue_colors_normal_split_fig1_epi]
ax_leg.legend(handles=handles, title="", loc="center",
              fontsize=14, frameon=False, ncol=1,
              handlelength=1, handleheight=0.8, labelspacing=0.3)

fig_leg.tight_layout()
fig_leg.savefig(os.path.join(output_dir, "legend_tissue_type_split.pdf"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
epi_dict = {'B-Cells': '#009432ff',
 'Endothelial': "#521243ff",
 'Fibroblasts': "#ff9090",
 'Mast Cells': "#A9422A",
 'Myeloid Cells': '#006266ff',
 'Pericytes': '#81fcf8',
 'Plasma Cells': '#f79f1fff',
 'Schwann Cells': '#8643cc',
 'Smooth Muscle': '#1518c2',
 'T-Cells': '#c4e538ff',
 'CD8-T-Cell':"#EA2027",
 'Epithelial': '#34ace0'}

palette_col1 = epi_dict
palette_col2 = cluster_colors

In [ ]:
# Cores legend
cell_counts = adata_filtered.obs["annotation_final_fine_cd8"].value_counts()
legend1_order = [k for k in cell_counts.index if k in palette_col1]
legend2_order = sorted(palette_col2.keys(), key=lambda x: int(x))

rename = {"CD8-T-Cell": "CD8-T-Cells"}

fig_leg, (ax_leg1, ax_leg2) = plt.subplots(1, 2, figsize=(10, 3))
ax_leg1.axis("off")
ax_leg2.axis("off")

handles1 = [mpatches.Patch(color=palette_col1[k], label=rename.get(k, k)) for k in legend1_order]
ax_leg1.legend(handles=handles1, title="", loc="center",
               fontsize=14, title_fontsize=14, frameon=False, ncol=2,
               handlelength=1, handleheight=0.8, labelspacing=0.3)

handles2 = [mpatches.Patch(color=palette_col2[k], label=k) for k in legend2_order]
ax_leg2.legend(handles=handles2, title="", loc="center",
               fontsize=14, title_fontsize=14, frameon=False, ncol=2,
               handlelength=1, handleheight=0.8, labelspacing=0.3)

fig_leg.tight_layout()
plt.savefig(os.path.join(output_dir, "all_cells_and_epi_clusts_legend.pdf"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
sc.settings._vector_friendly = True

ax = sc.pl.umap(adata_filtered, color="GDF15", show=False, size=2, title=None, legend_loc="none", cmap="RdYlBu_r", vmin="p1", vmax="p99")
ax.set_title("GDF15")
ax.set_axis_off()
plt.savefig(os.path.join(output_dir, "GDF15_all_cells_umap.pdf"), dpi=300, bbox_inches="tight")
plt.show()
sc.settings._vector_friendly = False

In [ ]:
# Figure 2 validation
epi_obs_f2 = epi_adata.obs[epi_adata.obs["annotation_final_coarse"] == "Epithelial"]

for col in ["tissue_type_cell_level", "tissue_type_dysplasia_cell_level"]:
    print(f"\n=== {col} ===")
    counts = epi_obs_f2.groupby(["core_id", col]).size().reset_index(name="n_cells")
    core_label_counts = counts.groupby("core_id").filter(lambda x: len(x) > 1)
    small_labels = core_label_counts[core_label_counts["n_cells"] < 10]
    if len(small_labels) > 0:
        print(f"WARNING: cores with a label < 10 cells:")
        print(small_labels.to_string(index=False))
    else:
        print("All labels in mixed cores have >= 10 cells.")

cell_ids = epi_adata.obs["cell_id"].astype(str)
n_total = len(cell_ids)
n_unique = cell_ids.nunique()
n_dupes = n_total - n_unique

if n_dupes > 0:
    dupe_ids = cell_ids[cell_ids.duplicated(keep=False)].unique().tolist()
    print(f"WARNING: {n_dupes} duplicate cell_ids found: {dupe_ids}")
    dupes = epi_adata.obs[cell_ids.isin(dupe_ids)]
    print(dupes[["cell_id", "core_id", "annotation_final_coarse", "tissue_type_cell_level"]].to_string())
else:
    print(f"No duplicate cell_ids found ({n_total} cells, all unique).")

cell_ids = epi_adata.obs["cell_id"].astype(str)
mask = ~cell_ids.duplicated(keep="first")
normal_adata = epi_adata[mask].copy()

print(f"Cells before: {len(mask)}")
print(f"Cells after:  {epi_adata.n_obs}")
print(f"Removed:      {(~mask).sum()}")

In [ ]:
tissue_colors_f2 = {'Adj_N': '#40407a', 'AD': '#ffb142', 'CA': '#b33939'}
tissue_colors_normal_split_f2 = {'Dist_N': "#33fd55",'Adj_N': '#40407a', 'AD': '#ffb142', 'CA': '#b33939'}

tissue_dysplasia_colors_normal_split = {'Dist_N': "#33fd55",'Adj_N': '#40407a', 'LGD':"#ffda79",'HGD':'#ff793f', 'CA': '#b33939'}
tissue_dysplasia_colors = {'Adj_N': '#40407a', 'LGD':"#ffda79",'HGD':'#ff793f', 'CA': '#b33939'}

TISSUE_PALETTE = tissue_colors_normal_split_f2
DYSPLASIA_PALETTE = tissue_dysplasia_colors_normal_split
TISSUE_ORDER    = ['Dist_N','Adj_N', 'AD', 'CA', 'NA']
DYSPLASIA_ORDER = ['Dist_N','Adj_N', 'LGD', 'HGD', 'CA', 'NA']

In [ ]:
def plot_genes_expressed(adata, gene_list, size, ncols, thresh=0, show_axes=False):
    color_list = []
    for gene_name in gene_list:
        if gene_name in adata.var_names:
            col_name = f'{gene_name} > {thresh} transcripts'
            if 'counts' in adata.layers:
                expr = adata[:, gene_name].layers['counts']
            elif adata.raw is not None:
                expr = adata.raw[:, gene_name].X
            else:
                expr = adata[:, gene_name].X
            if hasattr(expr, 'toarray'):
                expr = expr.toarray().flatten()
            else:
                expr = expr.flatten()
            adata.obs[col_name] = pd.Categorical(
                (expr > thresh).astype(str), categories=['False', 'True']
            )
            color_key = f'{col_name}_colors'
            if color_key in adata.uns:
                del adata.uns[color_key]
            color_list.append(col_name)
    if color_list:
        ax = sc.pl.umap(adata, color=color_list, size=size, ncols=ncols,
                        palette=['lightgray', 'red'], show=False)
        if not show_axes:
            axes = ax if isinstance(ax, list) else [ax]
            for a in axes:
                a.set_axis_off()
        plt.show()
    if color_list:
        sc.pl.umap(adata, color=color_list, size=size, ncols=ncols,
                   palette=['lightgray', 'red'])

In [ ]:
tissue_colors_f2_dup = {'Adj_N': '#40407a', 'AD': '#ffb142', 'CA': '#b33939'}
tissue_colors_normal_split_f2_dup = {'Dist_N': "#33fd55",'Adj_N': '#40407a', 'AD': '#ffb142', 'CA': '#b33939'}

tissue_dysplasia_colors_normal_split_dup = {'Dist_N': "#33fd55",'Adj_N': '#40407a', 'LGD':"#ffda79",'HGD':'#ff793f', 'CA': '#b33939'}
tissue_dysplasia_colors_dup = {'Adj_N': '#40407a', 'LGD':"#ffda79",'HGD':'#ff793f', 'CA': '#b33939'}

res = "epithelial_0.8"
output_dir = str(P.results.figures / "figure_2")
os.makedirs(output_dir, exist_ok=True)


## Stemness and senescence violin plots

In [ ]:
epi_adata.obs['GDF15 > 0 transcripts'] = (epi_adata[:, 'GDF15'].X.toarray().flatten() > 0).astype(str)

In [ ]:
def plot_combined_stemness_senescence_analysis_violin(
    adata,
    cluster_col: str,
    figsize: Tuple[int, int] = (10, 8),
    palette=None
):
    clusters = sorted(adata.obs[cluster_col].unique(), key=lambda x: int(x))
    if palette is None:
        colors = sns.color_palette('tab10', len(clusters))
    elif isinstance(palette, dict):
        colors = [palette[c] for c in clusters]
    elif isinstance(palette, list):
        colors = palette[:len(clusters)]
    else:
        colors = sns.color_palette(palette, len(clusters))
    color_dict = dict(zip(clusters, colors))

    fig, axes = plt.subplots(1, 3, figsize=figsize, sharey=True)

    gdf15_percentages = []
    for cluster in clusters:
        mask = adata.obs[cluster_col] == cluster
        pct = (adata.obs.loc[mask, 'GDF15 > 0 transcripts'] == 'True').sum() / mask.sum() * 100
        gdf15_percentages.append(pct)
    axes[0].barh(range(len(clusters)), gdf15_percentages, color=colors, alpha=0.8)
    axes[0].set_xlabel('% GDF15+ Cells', fontsize=13, fontweight='bold')
    axes[0].set_xlim(0, max(gdf15_percentages) * 1.1 if max(gdf15_percentages) > 0 else 10)
    axes[0].set_yticks(range(len(clusters)))
    axes[0].set_yticklabels(clusters)
    axes[0].invert_yaxis()
    axes[0].set_ylabel(cluster_col, fontsize=13, fontweight='bold')

    sns.violinplot(data=adata.obs, y=cluster_col, x='stemness_score',
                   ax=axes[1], palette=color_dict, order=clusters,
                   inner='box', cut=0, orient='h')
    axes[1].set_xlabel('Stemness Score', fontsize=13, fontweight='bold')

    sns.violinplot(data=adata.obs, y=cluster_col, x='senepy_intestine_epi_0',
                   ax=axes[2], palette=color_dict, order=clusters,
                   inner='box', cut=0, orient='h')
    axes[2].set_xlabel('SenePy Score', fontsize=13, fontweight='bold')

    for i, ax in enumerate(axes):
        if i > 0:
            ax.set_ylabel('')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(True, alpha=0.3, axis='x')

    plt.setp(axes[0].get_yticklabels(), fontsize=11, fontweight='bold')

    plt.tight_layout()
    plt.subplots_adjust(wspace=0.1)
    return fig, axes, {
        'gdf15_percentages': pd.Series(gdf15_percentages, index=clusters),
        'stemness_means': adata.obs.groupby(cluster_col)['stemness_score'].mean(),
        'senepy_means': adata.obs.groupby(cluster_col)['senepy_intestine_epi_0'].mean(),
    }

In [ ]:
fig, axes, summary = plot_combined_stemness_senescence_analysis_violin(
    epi_adata,
    cluster_col=res,
    palette=cluster_colors,
    figsize= (6, 10)
)

fig.savefig(os.path.join(output_dir, "combo_stem_senesc_gdf15_violin_vertical.pdf"), dpi=300, bbox_inches="tight")

## Marker genes

In [ ]:
markers = ["GDF15","CXCL1","CXCL2","CXCL3","AREG","CD55","CCL20","CDKN1A","OLFM4","SLC12A2","LGR5","HMGB1","TUBB","PCNA","MKI67","SLC26A2","SLC26A3","KRT20","FCGBP","TFF3", "RNF43","CCL5","ANXA2","CXCL14","APCDD1"]

dp = sc.pl.dotplot(
    epi_adata,
    markers,
    groupby=res,
    standard_scale="var",
    return_fig=True,
    swap_axes=True,
)
dp.style(dot_edge_color="none")
ax_dict = dp.show(return_axes=True)

ax_dict["mainplot_ax"].tick_params(axis="both", labelsize=14)
ax_dict["mainplot_ax"].set_xticklabels(
    ax_dict["mainplot_ax"].get_xticklabels(), rotation=0, ha="center", fontsize=14
)

for ax_name, ax in ax_dict.items():
    if ax_name != "mainplot_ax":
        ax.tick_params(labelsize=12)
        if ax.get_title():
            ax.set_title(ax.get_title(), fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "markers_dotplot.pdf"), dpi=300, bbox_inches="tight")
plt.show()

## Cluster composition by tissue

In [ ]:
def plot_cluster_composition_by_tissue(
    adata,
    cluster_col: str,
    cluster_palette: list = None,
    figsize: Tuple[int, int] = (8, 6),
):
    tissue_normal_split_order = ['Dist_N', 'Adj_N', 'AD', 'CA', 'NA']
    clusters = sorted(adata.obs[cluster_col].astype(str).unique(), key=lambda x: int(x))
    if cluster_palette is None:
        cluster_colors_local = dict(zip(clusters, sns.color_palette("tab20", len(clusters))))
    elif isinstance(cluster_palette, list):
        cluster_colors_local = dict(zip(clusters, cluster_palette[:len(clusters)]))
    else:
        cluster_colors_local = cluster_palette

    def make_bar_plot(ax, col, order, xlabel):
        composition = pd.crosstab(
            adata.obs[col],
            adata.obs[cluster_col].astype(str),
            normalize='index'
        ) * 100
        available = [cat for cat in order if cat in composition.index]
        composition = composition.reindex(index=available, columns=clusters, fill_value=0)
        bottom = np.zeros(len(available))
        for cluster in clusters:
            ax.bar(range(len(available)), composition[cluster],
                   bottom=bottom, color=cluster_colors_local[cluster],
                   label=cluster, alpha=0.8)
            bottom += composition[cluster].values
        ax.set_ylabel('Composition (%)', fontsize=14, fontweight='bold')
        ax.set_ylim(0, 100)
        ax.set_xlabel(xlabel, fontsize=14, fontweight='bold')
        ax.set_xticks(range(len(available)))
        ax.set_xticklabels(available, rotation=45, ha='right', fontsize=14, fontweight='bold')
        ax.tick_params(axis='y', labelsize=14)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        return composition

    fig1, ax1 = plt.subplots(figsize=figsize)
    composition_tissue_normal_split = make_bar_plot(
        ax1, 'tissue_type_cell_level_normal_split', tissue_normal_split_order, 'Tissue Type'
    )
    plt.tight_layout()
    plt.show()

    fig_legend, ax_legend = plt.subplots(figsize=(3, len(clusters) * 0.3))
    handles = [plt.Rectangle((0, 0), 1, 1, color=cluster_colors_local[c], alpha=0.8) for c in clusters]
    ax_legend.legend(handles, clusters, title="Cluster", fontsize=14,
                 title_fontsize=14, frameon=False, loc='center',
                 handletextpad=1.0, labelspacing=0.8, ncol=2)
    ax_legend.axis('off')
    plt.tight_layout()
    plt.show()

    return (
        {'fig1': fig1, 'fig_legend': fig_legend},
        {'ax1': ax1},
        {'tissue_normal_split_composition': composition_tissue_normal_split},
    )

figs, axes, compositions = plot_cluster_composition_by_tissue(
    epi_adata,
    cluster_col=res,
    cluster_palette=cluster_colors
)

figs['fig1'].savefig(os.path.join(output_dir, "tissue_type_composition_barplot.pdf"), dpi=300, bbox_inches="tight")
figs['fig_legend'].savefig(os.path.join(output_dir, "cluster_legend.pdf"), dpi=300, bbox_inches="tight")

## UMAPs

In [ ]:
sc.settings._vector_friendly = True
ax = sc.pl.umap(epi_adata, color="senepy_intestine_epi_0", cmap="RdYlBu_r", vmax="p99", vmin="p1",size=2, show=False, title="Senepy Score")
ax.set_axis_off()
plt.savefig(os.path.join(output_dir, "senepy_score_umap.pdf"), dpi=300, bbox_inches="tight")
plt.show()

ax = sc.pl.umap(epi_adata, color="stemness_score", cmap="RdYlBu_r", vmax="p99", vmin="p1",size=2, show=False, title="Stemness Score")
ax.set_axis_off()
plt.savefig(os.path.join(output_dir, "stemness_score_umap.pdf"), dpi=300, bbox_inches="tight")
plt.show()

sc.settings._vector_friendly = False

## Neighborhood analysis

In [ ]:
tissue_types = ['Dist_N', 'Adj_N', 'AD', 'CA']
MIN_CELLS_PER_CLUSTER = 0
MIN_CELLS_PER_CORE = 5

tissue_results = {}

for tissue in tissue_types:
    print(f"\nProcessing tissue type: {tissue}")
    tissue_adata = epi_adata[epi_adata.obs['tissue_type_cell_level_normal_split'] == tissue].copy()

    cluster_counts = tissue_adata.obs['epithelial_0.8'].value_counts()
    valid_clusters = cluster_counts[cluster_counts >= MIN_CELLS_PER_CLUSTER].index.tolist()
    removed_clusters = cluster_counts[cluster_counts < MIN_CELLS_PER_CLUSTER].index.tolist()
    if removed_clusters:
        print(f"  Removing clusters with < {MIN_CELLS_PER_CLUSTER} cells: {removed_clusters}")

    tissue_adata = tissue_adata[tissue_adata.obs['epithelial_0.8'].isin(valid_clusters)].copy()
    if tissue_adata.n_obs == 0:
        print(f"  No valid cells for {tissue}, skipping.")
        continue

    core_adatas = []
    for core in sorted(tissue_adata.obs['core_id'].unique()):
        core_mask = tissue_adata.obs['core_id'] == core
        core_adata = tissue_adata[core_mask].copy()
        if core_mask.sum() < MIN_CELLS_PER_CORE:
            continue
        sq.gr.spatial_neighbors(core_adata, coord_type="generic", spatial_key="spatial", delaunay=True)
        core_adatas.append(core_adata)

    if len(core_adatas) == 0:
        print(f"  No valid cores for {tissue}, skipping.")
        continue

    connectivities = block_diag([ca.obsp['spatial_connectivities'] for ca in core_adatas], format='csr')
    distances = block_diag([ca.obsp['spatial_distances'] for ca in core_adatas], format='csr')

    core_order = np.concatenate([
        np.where(tissue_adata.obs['core_id'] == ca.obs['core_id'].iloc[0])[0]
        for ca in core_adatas
    ])
    reorder = np.argsort(core_order)
    connectivities = connectivities[reorder][:, reorder]
    distances = distances[reorder][:, reorder]

    tissue_adata.obsp['spatial_connectivities'] = connectivities
    tissue_adata.obsp['spatial_distances'] = distances
    tissue_adata.uns['spatial_neighbors'] = {
        'connectivities_key': 'spatial_connectivities',
        'distances_key': 'spatial_distances',
        'params': {'coord_type': 'generic', 'delaunay': True}
    }

    local_clusters = sorted(tissue_adata.obs['epithelial_0.8'].unique(), key=lambda x: int(x))
    tissue_adata.obs['epithelial_0.8'] = pd.Categorical(
        tissue_adata.obs['epithelial_0.8'], categories=local_clusters, ordered=True
    )

    sq.gr.nhood_enrichment(tissue_adata, cluster_key="epithelial_0.8")
    tissue_results[tissue] = tissue_adata

valid_tissues = [t for t in tissue_types if t in tissue_results]
all_clusters_global = sorted(
    set().union(*[set(tissue_results[t].obs['epithelial_0.8'].unique()) for t in valid_tissues]),
    key=lambda x: int(x)
)
print(f"\nGlobal cluster order: {all_clusters_global}")


def get_enrichment_df(adata, global_clusters):
    zscore = adata.uns['epithelial_0.8_nhood_enrichment']['zscore']
    local_clusters = adata.obs['epithelial_0.8'].cat.categories.tolist()
    df = pd.DataFrame(zscore, index=local_clusters, columns=local_clusters)
    df = df.reindex(index=global_clusters, columns=global_clusters)
    return df

In [ ]:
# Combined-N neighborhood enrichment plot
combined_tissues = [
    ("Adj_N", ["Adj_N", "Dist_N"]),
    ("AD", ["AD"]),
    ("CA", ["CA"]),
]
combined_tissues = [
    (name, [k for k in keys if k in tissue_results])
    for name, keys in combined_tissues
]

n_tissues = len(combined_tissues)
widths = [1] * n_tissues + [0.05]
fig, axes_all = plt.subplots(1, n_tissues + 1, figsize=(8 * n_tissues, 7),
                              gridspec_kw={'width_ratios': widths})
axes = axes_all[:n_tissues]
cbar_ax = axes_all[-1]
vmin, vmax = -100, 100
n_clusters = len(all_clusters_global)

for idx, (ax, (display_name, tissue_keys)) in enumerate(zip(axes, combined_tissues)):
    dfs = [get_enrichment_df(tissue_results[tk], all_clusters_global) for tk in tissue_keys]
    if len(dfs) > 1:
        df = pd.concat(dfs).groupby(level=0).mean()
        df = df.reindex(index=dfs[0].index, columns=dfs[0].columns)
    else:
        df = dfs[0]

    sns.heatmap(
        df,
        ax=ax,
        cmap='RdBu_r',
        vmin=vmin,
        vmax=vmax,
        square=True,
        linewidths=0.5,
        linecolor='lightgray',
        cbar=(idx == n_tissues - 1),
        cbar_ax=cbar_ax if (idx == n_tissues - 1) else None,
        cbar_kws={'label': 'Z-score'} if (idx == n_tissues - 1) else {},
        annot=False,
        mask=df.isna()
    )
    ax.set_title(display_name, fontsize=16, fontweight='bold')
    ax.set_xlabel('Cluster', fontsize=14, labelpad=50)
    ax.set_ylabel('')
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.tick_params(axis='both', length=0)

    for i, label in enumerate(all_clusters_global):
        color = cluster_colors.get(str(label), 'grey')
        ax.add_patch(plt.Rectangle((-0.8, i), 0.6, 1, color=color, clip_on=False))
        ax.text(-1.0, i + 0.5, str(label), ha='right', va='center', fontsize=16, clip_on=False)

    for i, label in enumerate(all_clusters_global):
        color = cluster_colors.get(str(label), 'grey')
        ax.add_patch(plt.Rectangle((i, n_clusters + 0.1), 1, 0.6, color=color, clip_on=False))
        ax.text(i + 0.5, n_clusters + 1.0, str(label), ha='center', va='top', fontsize=16, clip_on=False)

axes[0].set_ylabel('Cluster', fontsize=14, labelpad=45)

plt.subplots_adjust(wspace=0.2)
plt.savefig(os.path.join(output_dir, "neighborhood_matrices_by_tissue_type.pdf"), dpi=300, bbox_inches="tight")
plt.show()